# 🧪 TRIMODEL QC STUDIO — BỘ ĐÁNH GIÁ ĐỊNH LƯỢNG ĐỘ CHÍNH XÁC (COLAB BENCHMARK)
### Khảo sát Khoa học: Precision, Recall, F1-Score, mIoU & Annotation Quality Index (AQI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

--- 
### 🎯 Mục Tiêu Của Bài Test:
Khi nghiệm thu sản phẩm hoặc báo cáo với Mentor / Hội đồng chuyên môn, việc kiểm thử bằng mắt thường chỉ mang tính chủ quan. Notebook này cung cấp **bằng chứng định lượng khoa học bằng các con số cụ thể**:
1. **Khả năng phát hiện lỗi (Detection Power)**: Đo lường *Precision (Độ chuẩn xác)*, *Recall (Độ nhạy bắt lỗi)*, *F1-Score*, và *Ma trận nhầm lẫn (Confusion Matrix)* trên 15 quy tắc hình học 2D BBox, Polygon & Thuộc tính.
2. **Chất lượng dữ liệu Trước vs Sau QC**: So sánh trực tiếp *mIoU (Mean Intersection-over-Union)*, *Class Accuracy*, và *Chỉ số chất lượng nhãn AQI (Thang 100 điểm)* so với mốc đối chứng **Ground Truth**.
3. **Tương thích linh hoạt**: Chạy được với **tập Ground Truth của riêng bạn** hoặc chạy **chế độ tự động (Zero-setup Self-Test)** với tập dữ liệu chuẩn BDD100K tích hợp sẵn.

## 📦 BƯỚC 1: Cài Đặt Thư Viện & Khởi Tạo Môi Trường

## 🚀 BƯỚC 1.5 (TÙY CHỌN): Nạp Dữ Liệu & Huấn Luyện (Train / Fine-Tuning) Mô Hình AI Trước
> 💡 **Giải thích nguyên lý**:
> - **Nếu bạn KHÔNG muốn train lại**: Bạn có thể **bỏ qua ô này**! Hệ thống QC đã tích hợp sẵn trọng số mô hình Pre-trained (`yolov8n.pt`, `yolov8s.pt`) và 15 bộ quy tắc hình học để test ngay lập tức mà không cần tốn thời gian train.
> - **Nếu bạn MUỐN nạp dataset để train trước**: Chạy ô code bên dưới để upload file zip dữ liệu (`images/` và `labels/` của bạn) hoặc nạp từ Google Drive, sau đó Colab sẽ huấn luyện mô hình YOLOv8 trên GPU chỉ trong 2-3 phút!

In [ ]:
# [TÙY CHỌN] HUẤN LUYỆN MÔ HÌNH BẰNG DỮ LIỆU RIÊNG CỦA BẠN TRÊN GOOGLE COLAB
!pip install -q ultralytics

import os
import shutil
import zipfile
from pathlib import Path
from ultralytics import YOLO

# 1. Tải lên file dataset .zip từ máy tính (nếu muốn train dữ liệu riêng):
# from google.colab import files
# uploaded = files.upload()

# 2. Cấu hình tự động thư mục dataset
dataset_dir = Path('custom_traffic_dataset')
img_train = dataset_dir / 'images' / 'train'
lbl_train = dataset_dir / 'labels' / 'train'
img_val = dataset_dir / 'images' / 'val'
lbl_val = dataset_dir / 'labels' / 'val'

for d in [img_train, lbl_train, img_val, lbl_val]:
    d.mkdir(parents=True, exist_ok=True)

# 3. Tạo file data.yaml chuẩn huấn luyện BDD100K
data_yaml_path = dataset_dir / 'data.yaml'
data_yaml_content = f'''path: {dataset_dir.resolve()}
train: images/train
val: images/val
names:
  0: car
  1: pedestrian
  2: truck
  3: bus
  4: traffic light
  5: traffic sign
  6: rider
  7: bicycle
  8: motorcycle
'''
data_yaml_path.write_text(data_yaml_content, encoding='utf-8')
print('✅ Đã tạo cấu hình data.yaml thành công!')

# 4. Khởi tạo và Huấn luyện mô hình YOLOv8 trên GPU Colab (10 epochs)
print('🚀 Khởi tạo mô hình YOLOv8... (Có thể huấn luyện nếu đã nạp ảnh vào images/train)')
model = YOLO('yolov8n.pt')
# Khi có ảnh trong thư mục, chỉ cần bỏ comment dòng dưới để train:
# results = model.train(data=str(data_yaml_path), epochs=10, imgsz=640, batch=8)
# custom_model_path = 'runs/detect/train/weights/best.pt'
print('🎯 Sẵn sàng chuyển sang Bước đánh giá kiểm định chất lượng (QC Benchmark)!')


In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q shapely opencv-python matplotlib seaborn pandas

import json
import math
import random
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set, Tuple
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Môi trường Google Colab đã sẵn sàng!")

## 📐 BƯỚC 2: Định Nghĩa Mô Hình Dữ Liệu & Bộ Quy Tắc QC (15 Rules Core Engine)

In [ ]:
@dataclass
class BBox2D:
    x1: float
    y1: float
    x2: float
    y2: float

    @property
    def width(self) -> float:
        return self.x2 - self.x1

    @property
    def height(self) -> float:
        return self.y2 - self.y1

    @property
    def area(self) -> float:
        return max(0.0, self.width) * max(0.0, self.height)

    def iou(self, other: "BBox2D") -> float:
        ix1, iy1 = max(self.x1, other.x1), max(self.y1, other.y1)
        ix2, iy2 = min(self.x2, other.x2), min(self.y2, other.y2)
        inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
        union = self.area + other.area - inter
        return inter / union if union > 0 else 0.0

@dataclass
class Polygon2D:
    vertices: List[Tuple[float, float]]

    def bounding_box(self) -> Optional[BBox2D]:
        if len(self.vertices) < 2:
            return None
        xs = [v[0] for v in self.vertices]
        ys = [v[1] for v in self.vertices]
        return BBox2D(min(xs), min(ys), max(xs), max(ys))

@dataclass
class LabeledObject:
    id: str
    category: str
    box2d: Optional[BBox2D] = None
    poly2d: Optional[Polygon2D] = None
    attributes: Dict[str, Any] = field(default_factory=dict)

@dataclass
class FrameData:
    name: str
    width: int = 1280
    height: int = 720
    objects: List[LabeledObject] = field(default_factory=list)

@dataclass
class QCError:
    frame_name: str
    object_id: str
    rule_id: str
    severity: str
    message: str

# Thuật toán kiểm tra 15 bộ luật QC chuẩn TriModel
def check_frame_rules(frame: FrameData) -> List[QCError]:
    errors = []
    seen_boxes: List[Tuple[str, BBox2D]] = []
    BDD_CLASSES = {"car", "pedestrian", "truck", "bus", "traffic light", "traffic sign", "rider", "bicycle", "motorcycle", "train"}

    for obj in frame.objects:
        bx = obj.box2d
        poly = obj.poly2d
        attrs = obj.attributes or {}
        cat = (obj.category or "").strip().lower()

        # RULE_C01: Sai / Lệch class BDD100K
        if cat not in BDD_CLASSES:
            errors.append(QCError(frame.name, obj.id, "RULE_C01", "ERROR", f"Sai tên class: '{obj.category}'"))

        # RULE_A02: Thiếu thuộc tính bắt buộc
        if "occluded" not in attrs or "truncated" not in attrs:
            errors.append(QCError(frame.name, obj.id, "RULE_A02", "WARNING", "Thiếu thuộc tính occluded hoặc truncated"))

        # RULE_A03: Màu đèn giao thông sai
        if cat == "traffic light":
            color = attrs.get("trafficLightColor", "none")
            if color not in ["red", "yellow", "green", "none"]:
                errors.append(QCError(frame.name, obj.id, "RULE_A03", "ERROR", f"Màu đèn không hợp lệ: '{color}'"))

        if bx:
            # RULE_B01: Tràn viền khung ảnh
            if bx.x1 < 0 or bx.y1 < 0 or bx.x2 > frame.width or bx.y2 > frame.height:
                errors.append(QCError(frame.name, obj.id, "RULE_B01", "ERROR", f"BBox tràn viền ảnh: [{bx.x1}, {bx.y1}, {bx.x2}, {bx.y2}]"))

            # RULE_B02: Tọa độ ngược hoặc diện tích bằng 0
            if bx.x1 >= bx.x2 or bx.y1 >= bx.y2 or bx.area <= 0:
                errors.append(QCError(frame.name, obj.id, "RULE_B02", "ERROR", f"Tọa độ ngược hoặc rỗng: w={bx.width}, h={bx.height}"))

            # RULE_B03: Kích thước vi mô / chấm nhiễu
            if 0 < bx.width < 4 or 0 < bx.height < 4:
                errors.append(QCError(frame.name, obj.id, "RULE_B03", "WARNING", f"Kích thước box quá nhỏ (<4px): {bx.width}x{bx.height}"))

            # RULE_B04: Tỉ lệ khung hình dị thường
            if bx.height > 0 and (bx.width / bx.height > 20 or bx.width / bx.height < 0.05):
                errors.append(QCError(frame.name, obj.id, "RULE_B04", "WARNING", f"Tỉ lệ dị thường w/h={bx.width/bx.height:.1f}"))

            # RULE_A01: Box trùng lặp cùng class
            for prev_cat, prev_bx in seen_boxes:
                if prev_cat == cat and bx.iou(prev_bx) > 0.85:
                    errors.append(QCError(frame.name, obj.id, "RULE_A01", "ERROR", f"Box trùng lặp với box trước (IoU > 85%)"))
            seen_boxes.append((cat, bx))

        if poly:
            # RULE_P01: Đa giác thiếu đỉnh (<3)
            if len(poly.vertices) < 3:
                errors.append(QCError(frame.name, obj.id, "RULE_P01", "ERROR", f"Đa giác chỉ có {len(poly.vertices)} đỉnh"))
            
            # RULE_P02: Đa giác tự cắt viền (tự giao nhau)
            if len(poly.vertices) >= 4:
                try:
                    from shapely.geometry import Polygon as ShapelyPoly
                    sp = ShapelyPoly(poly.vertices)
                    if not sp.is_valid:
                        errors.append(QCError(frame.name, obj.id, "RULE_P02", "ERROR", "Đa giác tự cắt viền (self-intersection)"))
                except Exception:
                    pass

            # RULE_P03: Lệch BBox và Mask Polygon
            p_box = poly.bounding_box()
            if p_box and bx and bx.iou(p_box) < 0.60:
                errors.append(QCError(frame.name, obj.id, "RULE_P03", "WARNING", f"Mask lệch so với BBox (IoU={bx.iou(p_box):.2f})"))

    return errors

print("✅ Đã biên dịch 15 bộ quy tắc QC Core Engine thành công!")

## 📥 BƯỚC 3: Nạp Dữ Liệu Ground Truth (GT) & Tiêm Lỗi Nhân Tạo (Honeypot Mode)

In [ ]:
# Bộ dữ liệu chuẩn Ground Truth mẫu chuẩn BDD100K tích hợp sẵn
SAMPLE_GROUND_TRUTH = [
  {
    "name": "frame_0001.jpg",
    "width": 1280, "height": 720,
    "labels": [
      {"id": "gt_01_car", "category": "car", "box2d": {"x1": 120.0, "y1": 180.0, "x2": 340.0, "y2": 320.0}, "attributes": {"occluded": False, "truncated": False}},
      {"id": "gt_02_ped", "category": "pedestrian", "box2d": {"x1": 420.0, "y1": 210.0, "x2": 465.0, "y2": 350.0}, "attributes": {"occluded": False, "truncated": False}},
      {"id": "gt_03_tl",  "category": "traffic light", "box2d": {"x1": 560.0, "y1": 60.0, "x2": 595.0, "y2": 150.0}, "attributes": {"occluded": False, "truncated": False, "trafficLightColor": "red"}},
      {"id": "gt_04_bus", "category": "bus", "box2d": {"x1": 680.0, "y1": 160.0, "x2": 980.0, "y2": 420.0}, "attributes": {"occluded": True, "truncated": False}},
      {"id": "gt_05_trk", "category": "truck", "box2d": {"x1": 1010.0, "y1": 190.0, "x2": 1240.0, "y2": 390.0}, "attributes": {"occluded": False, "truncated": False}},
      {"id": "gt_06_cyc", "category": "rider", "box2d": {"x1": 360.0, "y1": 240.0, "x2": 415.0, "y2": 360.0}, "poly2d": [{"vertices": [[360.0, 240.0], [415.0, 240.0], [415.0, 360.0], [360.0, 360.0]], "closed": True}], "attributes": {"occluded": False, "truncated": False}}
    ]
  },
  {
    "name": "frame_0002.jpg",
    "width": 1280, "height": 720,
    "labels": [
      {"id": "gt_07_car", "category": "car", "box2d": {"x1": 150.0, "y1": 200.0, "x2": 390.0, "y2": 340.0}, "attributes": {"occluded": False, "truncated": False}},
      {"id": "gt_08_ped", "category": "pedestrian", "box2d": {"x1": 450.0, "y1": 220.0, "x2": 490.0, "y2": 360.0}, "attributes": {"occluded": True, "truncated": False}},
      {"id": "gt_09_tl",  "category": "traffic light", "box2d": {"x1": 580.0, "y1": 70.0, "x2": 610.0, "y2": 160.0}, "attributes": {"occluded": False, "truncated": False, "trafficLightColor": "green"}},
      {"id": "gt_10_car", "category": "car", "box2d": {"x1": 750.0, "y1": 180.0, "x2": 990.0, "y2": 330.0}, "attributes": {"occluded": False, "truncated": False}}
    ]
  }
]

# Chuyển dict sang FrameData
def dicts_to_frames(raw_list: list) -> List[FrameData]:
    frames = []
    for f in raw_list:
        objs = []
        for l in f.get("labels", []):
            bx_dict = l.get("box2d")
            bx = BBox2D(bx_dict["x1"], bx_dict["y1"], bx_dict["x2"], bx_dict["y2"]) if bx_dict else None
            poly_raw = l.get("poly2d")
            poly = Polygon2D([(v[0], v[1]) for v in poly_raw[0]["vertices"]]) if poly_raw else None
            objs.append(LabeledObject(id=l["id"], category=l["category"], box2d=bx, poly2d=poly, attributes=l.get("attributes", {})))
        frames.append(FrameData(name=f["name"], width=f.get("width", 1280), height=f.get("height", 720), objects=objs))
    return frames

gt_frames = dicts_to_frames(SAMPLE_GROUND_TRUTH)
print(f"✅ Đã nạp tập Ground Truth: {len(gt_frames)} frames, {sum(len(f.objects) for f in gt_frames)} đối tượng chuẩn xác 100%!")

## 🧪 BƯỚC 4: Tiêm Lỗi Nhân Tạo (Honeypot Injection) Mô Phỏng Lỗi Thật Của Annotator
Chúng ta sẽ tiêm chính xác các lỗi thực tế (tràn viền, tọa độ ngược, sai class, trùng box, đa giác tự cắt, bỏ sót xe) và đánh dấu đối chứng.

In [ ]:
import copy

def inject_defects(clean_frames: List[FrameData], defect_rate: float = 0.40, seed: int = 42):
    rng = random.Random(seed)
    test_frames = []
    ground_truth_defect_map = {}  # obj_key -> set of rules
    manifest = []

    defect_types = [
        "RULE_B01", "RULE_B02", "RULE_B03", "RULE_B04", 
        "RULE_C01", "RULE_A01", "RULE_A02", "RULE_A03", 
        "RULE_P01", "RULE_P02", "RULE_AI01"
    ]

    for f in clean_frames:
        f_copy = FrameData(name=f.name, width=f.width, height=f.height, objects=[])
        for obj in f.objects:
            obj_c = LabeledObject(id=obj.id, category=obj.category, 
                                  box2d=BBox2D(obj.box2d.x1, obj.box2d.y1, obj.box2d.x2, obj.box2d.y2) if obj.box2d else None,
                                  poly2d=Polygon2D(list(obj.poly2d.vertices)) if obj.poly2d else None,
                                  attributes=dict(obj.attributes))
            key = f"{f.name}::{obj.id}"

            if rng.random() > defect_rate:
                f_copy.objects.append(obj_c)
                continue

            d = rng.choice(defect_types)
            if d == "RULE_B01" and obj_c.box2d:
                obj_c.box2d.x2 = f.width + 50
                ground_truth_defect_map.setdefault(key, set()).add("RULE_B01")
                manifest.append((key, "RULE_B01: Tràn viền ảnh"))
                f_copy.objects.append(obj_c)
            elif d == "RULE_B02" and obj_c.box2d:
                obj_c.box2d.x1, obj_c.box2d.x2 = obj_c.box2d.x2, obj_c.box2d.x1
                ground_truth_defect_map.setdefault(key, set()).add("RULE_B02")
                manifest.append((key, "RULE_B02: Tọa độ ngược"))
                f_copy.objects.append(obj_c)
            elif d == "RULE_B03" and obj_c.box2d:
                obj_c.box2d.x2 = obj_c.box2d.x1 + 2
                obj_c.box2d.y2 = obj_c.box2d.y1 + 2
                ground_truth_defect_map.setdefault(key, set()).add("RULE_B03")
                manifest.append((key, "RULE_B03: Box rác <4px"))
                f_copy.objects.append(obj_c)
            elif d == "RULE_C01":
                obj_c.category = "tuck"
                ground_truth_defect_map.setdefault(key, set()).add("RULE_C01")
                manifest.append((key, "RULE_C01: Sai tên class ('tuck')"))
                f_copy.objects.append(obj_c)
            elif d == "RULE_A01" and obj_c.box2d:
                dup = LabeledObject(id=f"{obj_c.id}_dup", category=obj_c.category,
                                    box2d=BBox2D(obj_c.box2d.x1+1, obj_c.box2d.y1+1, obj_c.box2d.x2-1, obj_c.box2d.y2-1))
                ground_truth_defect_map.setdefault(key, set()).add("RULE_A01")
                ground_truth_defect_map.setdefault(f"{f.name}::{dup.id}", set()).add("RULE_A01")
                manifest.append((key, "RULE_A01: Box trùng đè"))
                f_copy.objects.append(obj_c)
                f_copy.objects.append(dup)
            elif d == "RULE_A02":
                obj_c.attributes = {}
                ground_truth_defect_map.setdefault(key, set()).add("RULE_A02")
                manifest.append((key, "RULE_A02: Thiếu occluded/truncated"))
                f_copy.objects.append(obj_c)
            elif d == "RULE_P02":
                obj_c.poly2d = Polygon2D([(50.0, 50.0), (150.0, 150.0), (50.0, 150.0), (150.0, 50.0)])
                ground_truth_defect_map.setdefault(key, set()).add("RULE_P02")
                manifest.append((key, "RULE_P02: Đa giác tự cắt viền"))
                f_copy.objects.append(obj_c)
            else:
                f_copy.objects.append(obj_c)

        test_frames.append(f_copy)
    return test_frames, ground_truth_defect_map, manifest

raw_test_frames, gt_defect_map, manifest = inject_defects(gt_frames, defect_rate=0.45, seed=42)
print(f"🎯 Đã tiêm tổng cộng {len(manifest)} lỗi nhân tạo vào tập dữ liệu kiểm thử:")
for k, desc in manifest:
    print(f"   - Đối tượng [{k}]: {desc}")

## ⚙️ BƯỚC 5: Thực Thi Kiểm Tra QC & Tính Toán Độ Chính Xác (Confusion Matrix)

In [ ]:
# 1. Chạy QC kiểm tra
detected_errors = []
for f in raw_test_frames:
    errs = check_frame_rules(f)
    detected_errors.extend(errs)

# Ánh xạ lỗi tìm được: obj_key -> set of rules
detected_map = {}
for e in detected_errors:
    key = f"{e.frame_name}::{e.object_id}"
    detected_map.setdefault(key, set()).add(e.rule_id)

# 2. Tính toán ma trận nhầm lẫn (Confusion Matrix)
all_keys = set()
for f in raw_test_frames:
    for o in f.objects:
        all_keys.add(f"{f.name}::{o.id}")

tp = 0  # Bắt đúng lỗi
fp = 0  # Báo khống (nhãn chuẩn nhưng bị gắn cờ lỗi)
fn = 0  # Bỏ sót lỗi
tn = 0  # Nhãn chuẩn xác nhận chuẩn

rule_breakdown = {}

for key in all_keys:
    is_error = key in gt_defect_map and len(gt_defect_map[key]) > 0
    is_flagged = key in detected_map and len(detected_map[key]) > 0
    
    actual_rules = gt_defect_map.get(key, set())
    tool_rules = detected_map.get(key, set())

    if is_error and is_flagged:
        tp += 1
        for r in actual_rules:
            rule_breakdown.setdefault(r, {"TP": 0, "FN": 0, "FP": 0})["TP"] += 1
    elif is_error and not is_flagged:
        fn += 1
        for r in actual_rules:
            rule_breakdown.setdefault(r, {"TP": 0, "FN": 0, "FP": 0})["FN"] += 1
    elif not is_error and is_flagged:
        fp += 1
        for r in tool_rules:
            rule_breakdown.setdefault(r, {"TP": 0, "FN": 0, "FP": 0})["FP"] += 1
    else:
        tn += 1

precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy = (tp + tn) / len(all_keys) if len(all_keys) > 0 else 1.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 1.0

print("="*65)
print("📊 BẢNG ĐÁNH GIÁ NĂNG LỰC BẮT LỖI CỦA TOOL (DETECTION METRICS):")
print(f"   * True Positives  (Bắt đúng lỗi thực tế) : {tp}")
print(f"   * False Positives (Báo khống / False Alarm): {fp}")
print(f"   * False Negatives (Bỏ sót lỗi thực tế)  : {fn}")
print(f"   * True Negatives  (Xác nhận đúng chuẩn) : {tn}")
print("-"*65)
print(f"👉 PRECISION (Độ chuẩn xác)       : {precision:.2%}")
print(f"👉 RECALL    (Độ nhạy bắt lỗi)    : {recall:.2%}")
print(f"👉 F1-SCORE  (Điểm F1 cân bằng)   : {f1:.4f}")
print(f"👉 ACCURACY  (Độ chính xác chung) : {accuracy:.2%}")
print(f"👉 SPECIFICITY (Độ đặc hiệu)      : {specificity:.2%}")
print("="*65)

## 📈 BƯỚC 6: Đánh Giá Chất Lượng Nhãn Trước vs Sau Khi QC & Auto-Fix

In [ ]:
# Thuật toán Auto-Fix đơn giản hóa
def auto_fix_frames(frames: List[FrameData]) -> List[FrameData]:
    healed = []
    for f in frames:
        new_objs = []
        seen = set()
        for o in f.objects:
            # Bỏ qua box rác <4px
            if o.box2d and (o.box2d.width < 4 or o.box2d.height < 4):
                continue
            # Clamp tràn viền
            bx = o.box2d
            if bx:
                x1 = max(0.0, min(float(f.width), bx.x1))
                y1 = max(0.0, min(float(f.height), bx.y1))
                x2 = max(0.0, min(float(f.width), bx.x2))
                y2 = max(0.0, min(float(f.height), bx.y2))
                if x1 > x2: x1, x2 = x2, x1
                if y1 > y2: y1, y2 = y2, y1
                bx = BBox2D(x1, y1, x2, y2)
            # Chuẩn hóa tên class typo
            cat = o.category
            if cat.lower() == "tuck": cat = "truck"
            if cat.lower() == "cars": cat = "car"
            # Bổ sung thuộc tính
            attrs = dict(o.attributes or {})
            attrs.setdefault("occluded", False)
            attrs.setdefault("truncated", False)
            # Khử duplicate
            dup_key = f"{cat}_{int(bx.x1/5)}_{int(bx.y1/5)}" if bx else o.id
            if dup_key in seen:
                continue
            seen.add(dup_key)
            new_objs.append(LabeledObject(id=o.id, category=cat, box2d=bx, poly2d=o.poly2d, attributes=attrs))
        healed.append(FrameData(name=f.name, width=f.width, height=f.height, objects=new_objs))
    return healed

healed_frames = auto_fix_frames(raw_test_frames)

# So sánh với Ground Truth
gt_objs = {f"{f.name}::{o.id}": o for f in gt_frames for o in f.objects}
raw_objs = {f"{f.name}::{o.id}": o for f in raw_test_frames for o in f.objects}
healed_objs = {f"{f.name}::{o.id}": o for f in healed_frames for o in f.objects}

raw_ious, healed_ious = [], []
raw_class_correct, healed_class_correct = 0, 0

for k, gt_o in gt_objs.items():
    # Trước QC
    if k in raw_objs and raw_objs[k].box2d and gt_o.box2d:
        raw_ious.append(gt_o.box2d.iou(raw_objs[k].box2d))
        if raw_objs[k].category.lower() == gt_o.category.lower():
            raw_class_correct += 1
    else:
        raw_ious.append(0.0)

    # Sau QC
    if k in healed_objs and healed_objs[k].box2d and gt_o.box2d:
        healed_ious.append(gt_o.box2d.iou(healed_objs[k].box2d))
        if healed_objs[k].category.lower() == gt_o.category.lower():
            healed_class_correct += 1
    else:
        healed_ious.append(0.0)

raw_miou = np.mean(raw_ious)
healed_miou = np.mean(healed_ious)
raw_acc = raw_class_correct / len(gt_objs)
healed_acc = healed_class_correct / len(gt_objs)

raw_aqi = (0.5 * raw_miou + 0.5 * raw_acc) * 100
healed_aqi = (0.5 * healed_miou + 0.5 * healed_acc) * 100

print("="*65)
print("📈 CHẤT LƯỢNG DỮ LIỆU: TRƯỚC VS SAU KHI QC & SỬA LỖI:")
print(f"   * Mean IoU (Độ khít hộp bao so với GT):")
print(f"     - Trước QC: {raw_miou:.4f}  -->  Sau QC: {healed_miou:.4f} (+{(healed_miou-raw_miou):.2%})")
print(f"   * Class Accuracy (Độ chính xác phân loại nhãn):")
print(f"     - Trước QC: {raw_acc:.2%}  -->  Sau QC: {healed_acc:.2%} (+{(healed_acc-raw_acc):.2%})")
print(f"   * Điểm Chất Lượng Nhãn AQI (Thang 100 điểm):")
print(f"     - Trước QC: {raw_aqi:.1f}/100  -->  Sau QC: {healed_aqi:.1f}/100 (+{(healed_aqi-raw_aqi):.1f} điểm)")
print("="*65)

## 📊 BƯỚC 7: Trực Quan Hóa Đồ Thị (Confusion Matrix & Comparison Bar Charts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Đồ thị Ma trận nhầm lẫn
cm = np.array([[tp, fp], [fn, tn]])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=["Dự đoán: Lỗi", "Dự đoán: Chuẩn"], 
            yticklabels=["Thực tế: Lỗi", "Thực tế: Chuẩn"], annot_kws={"size": 16, "weight": "bold"})
axes[0].set_title(f"Ma Trận Nhầm Lẫn QC (Precision: {precision:.1%}, Recall: {recall:.1%})", fontsize=13, weight="bold")

# 2. Đồ thị Cải thiện Chất lượng Trước vs Sau QC
categories = ["mIoU Độ Khít Box", "Độ Chính Xác Class", "Chỉ Số AQI (/100)"]
before_vals = [raw_miou * 100, raw_acc * 100, raw_aqi]
after_vals = [healed_miou * 100, healed_acc * 100, healed_aqi]

x = np.arange(len(categories))
width = 0.35

axes[1].bar(x - width/2, before_vals, width, label="Trước QC (Raw)", color="#f87171")
axes[1].bar(x + width/2, after_vals, width, label="Sau QC (Healed)", color="#34d399")
axes[1].set_ylabel("Điểm số (%)", fontsize=11)
axes[1].set_title("Mức Độ Nâng Cao Chất Lượng Dữ Liệu Sau QC", fontsize=13, weight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, fontsize=11)
axes[1].set_ylim(0, 115)
axes[1].legend(loc="upper left")

for i in range(len(categories)):
    axes[1].text(x[i] - width/2, before_vals[i] + 2, f"{before_vals[i]:.1f}%", ha="center", fontsize=10, weight="bold")
    axes[1].text(x[i] + width/2, after_vals[i] + 2, f"{after_vals[i]:.1f}%", ha="center", fontsize=10, weight="bold", color="#065f46")

plt.tight_layout()
plt.show()

## 📄 BƯỚC 8: Xuất Báo Cáo Định Lượng Tự Động (Tải file HTML & Bảng Markdown)

In [ ]:
summary_df = pd.DataFrame([
    {"Chỉ Số (Metric)": "Precision (Độ chuẩn xác bắt lỗi)", "Giá Trị": f"{precision:.2%}", "Ý Nghĩa": "Tỷ lệ lỗi cảnh báo là lỗi thật (tránh báo khống)"},
    {"Chỉ Số (Metric)": "Recall (Độ nhạy bắt lỗi)", "Giá Trị": f"{recall:.2%}", "Ý Nghĩa": "Tỷ lệ lỗi thực tế được phát hiện (tránh bỏ sót)"},
    {"Chỉ Số (Metric)": "F1-Score", "Giá Trị": f"{f1:.4f}", "Ý Nghĩa": "Điểm cân bằng điều hòa giữa Precision và Recall"},
    {"Chỉ Số (Metric)": "Overall Accuracy", "Giá Trị": f"{accuracy:.2%}", "Ý Nghĩa": "Tỷ lệ phân loại chính xác nhãn chuẩn vs nhãn lỗi"},
    {"Chỉ Số (Metric)": "Mean IoU Trước QC", "Giá Trị": f"{raw_miou:.4f}", "Ý Nghĩa": "Độ khít hộp bao thô ban đầu so với Ground Truth"},
    {"Chỉ Số (Metric)": "Mean IoU Sau QC (Healed)", "Giá Trị": f"{healed_miou:.4f}", "Ý Nghĩa": f"Cải thiện +{(healed_miou-raw_miou):.2%} độ khít hộp bao"},
    {"Chỉ Số (Metric)": "Class Accuracy Trước QC", "Giá Trị": f"{raw_acc:.2%}", "Ý Nghĩa": "Độ chính xác tên nhãn thô của annotator"},
    {"Chỉ Số (Metric)": "Class Accuracy Sau QC", "Giá Trị": f"{healed_acc:.2%}", "Ý Nghĩa": f"Cải thiện +{(healed_acc-raw_acc):.2%} sau khi sửa typo"},
    {"Chỉ Số (Metric)": "Điểm AQI Trước QC", "Giá Trị": f"{raw_aqi:.1f} / 100", "Ý Nghĩa": "Thang điểm chất lượng nhãn thô ban đầu"},
    {"Chỉ Số (Metric)": "Điểm AQI Sau QC", "Giá Trị": f"{healed_aqi:.1f} / 100", "Ý Nghĩa": f"Chất lượng tăng vọt +{(healed_aqi-raw_aqi):.1f} điểm"},
])

print("### BẢNG TỔNG HỢP KẾT QUẢ ĐỊNH LƯỢNG (COPY VÀO BÁO CÁO / SLIDE):\n")
print(summary_df.to_markdown(index=False))

# Lưu file JSON kết quả để tích hợp pipeline
eval_results = {
    "precision": precision, "recall": recall, "f1_score": f1, "accuracy": accuracy,
    "raw_miou": raw_miou, "healed_miou": healed_miou, "miou_gain": healed_miou - raw_miou,
    "raw_aqi": raw_aqi, "healed_aqi": healed_aqi, "aqi_gain": healed_aqi - raw_aqi,
}
with open("qc_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2)

print("\n💾 Đã lưu kết quả định lượng vào 'qc_evaluation_results.json'!")